In [2]:
import pandas as pd
from tqdm import tqdm

## Analyze Pfam based on all parameters

In [10]:
accessions= ['at6923','at6929','at7143','at8285','at9104','at9336','at9503','at9578','at9744','at9762','at9806','at9830','at9847','at9852','at9879','at9883','at9900','col0']
#runs = ["","_cat","_no_co"]
runs = ["_cat"]
for run in runs:
    pfam_nlrs=pd.DataFrame()
    pfam_dom_gene=pd.DataFrame()
    pfam_dom_iso=pd.DataFrame()
    pfam_orf_gene=pd.DataFrame()
    pfam_orf_iso=pd.DataFrame()
    for acc in tqdm(accessions):
        # data preparation
        pfam = pd.read_csv(f"output/n5_genes_all/{acc}{run}/{acc}{run}_nlr_like.prot.fasta.tsv", sep="\t", names=["Query protein","Sequence MD5 digest","Sequence length","Analysis","Signature accession","Signature description","Start location","Stop location","Score (e-value)","Status (T: true)","Date","InterPro annotations - accession","InterPro annotations - description","GOTerms"])
        pfam = pfam.loc[pfam['Analysis']=="Pfam"]
        pfam["gene"]=pfam["Query protein"].str.split(".").str[0]
        pfam["isoform"]=pfam["Query protein"].str.split(".").str[0]+"."+pfam["Query protein"].str.split(".").str[1]

        # n query proteins
        # n query isoforms
        # n query genes

        ## NB-ARC analysis
        # get all transdecoder orfs with NB-ARC relation
        nbarc_trans = pfam.loc[pfam["Signature description"]=="NB-ARC domain"]["Query protein"].tolist()
        # extract full info from Pfam
        nbarc = pfam.loc[pfam["Query protein"].isin(nbarc_trans)]
        # NB-ARC gene:
        nlr_genes_dict={"gene":[],"TIR":[],"LRR":[],"CC":[]}
        #print(nlr_genes_dict)
        for i in nbarc.groupby("gene"):
            # get gene id
            nlr_genes_dict["gene"].append(i[0])

            # get TIR
            if "TIR domain" in i[1]["Signature description"].tolist():
                nlr_genes_dict["TIR"].append(1)
            else:
                nlr_genes_dict["TIR"].append(0)

            # get LRR
            lrr_signature=i[1]["Signature description"]
            lrr=0
            for j in lrr_signature:
                if "Leucine" in j:
                    lrr+=1
                    break
            nlr_genes_dict["LRR"].append(lrr)

            # get CC (RX domain in pfam!)

            cc_signature=i[1]["Signature description"]
            cc=0
            for j in cc_signature:
                if "Rx" in j:
                    cc+=1
                    break
            nlr_genes_dict["CC"].append(cc)
        nlr_out=pd.DataFrame.from_dict(nlr_genes_dict, orient="index").transpose()
        nlr_out["accession"]=acc
        pfam_nlrs=pd.concat([pfam_nlrs, nlr_out])


        ## Pfam analysis
        # gene
        dom_gene=pfam[["gene","InterPro annotations - description"]].drop_duplicates()
        dom_gene=dom_gene["InterPro annotations - description"].value_counts()
        dom_gene.rename(acc, inplace=True)
        pfam_dom_gene=pd.concat([pfam_dom_gene, dom_gene.head(n=11)], axis=1, sort=False)

        # isoform
        dom_iso=pfam[["isoform","InterPro annotations - description"]].drop_duplicates()
        dom_iso=dom_iso["InterPro annotations - description"].value_counts()
        dom_iso.rename(acc, inplace=True)
        pfam_dom_iso=pd.concat([pfam_dom_iso, dom_iso.head(n=11)], axis=1, sort=False)

        ## ORF analysis
        # gene
        orf_gene=pfam[["gene","GOTerms"]].drop_duplicates()
        orf_gene["GOTerms_list"]=orf_gene["GOTerms"].str.split("|")
        orf_gene=orf_gene.explode("GOTerms_list").drop("GOTerms", axis=1).drop_duplicates()["GOTerms_list"].value_counts()
        orf_gene.rename(acc, inplace=True)
        pfam_orf_gene=pd.concat([pfam_orf_gene, orf_gene.head(n=11)], axis=1, sort=False)

        # isoform
        orf_iso=pfam[["isoform","GOTerms"]].drop_duplicates()
        orf_iso["GOTerms_list"]=orf_iso["GOTerms"].str.split("|")
        orf_iso=orf_iso.explode("GOTerms_list").drop("GOTerms", axis=1).drop_duplicates()["GOTerms_list"].value_counts()
        orf_iso.rename(acc, inplace=True)
        pfam_orf_iso=pd.concat([pfam_orf_iso, orf_iso.head(n=11)], axis=1, sort=False)
    pfam_nlrs.to_csv(f"output/n5_genes_all/pfam{run}_nlrs.tsv", sep="\t")
    pfam_dom_gene.to_csv(f"output/n5_genes_all/pfam{run}_dom_gene.tsv", sep="\t")
    pfam_dom_iso.to_csv(f"output/n5_genes_all/pfam{run}_dom_iso.tsv", sep="\t")
    pfam_orf_gene.to_csv(f"output/n5_genes_all/pfam{run}_orf_gene.tsv", sep="\t")
    pfam_orf_iso.to_csv(f"output/n5_genes_all/pfam{run}_orf_iso.tsv", sep="\t")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [00:02<00:00,  7.40it/s]


### Print NLR like genes

In [3]:
nlr_domains=pd.read_csv("output/n5_genes_all/pfam_cat_nlrs.tsv", sep="\t")#.set_index("Unnamed: 0")

In [4]:
nlr_domains.drop("Unnamed: 0", axis=1).stack()

0   gene         G15278
    TIR               0
    LRR               1
    CC                0
    accession    at6923
                  ...  
15  gene          G2298
    TIR               1
    LRR               1
    CC                0
    accession    at9900
Length: 80, dtype: object

In [10]:
nlr_domains

,Unnamed: 0,gene,TIR,LRR,CC,accession
0,0,G15278,0,1,0,at6923
1,0,G3063,0,1,0,at6929
2,0,G11392,0,1,0,at7143
3,1,G14874,1,1,0,at7143
4,0,G5296,1,1,0,at9104
5,0,G11119,1,1,0,at9503
6,0,G15658,0,1,0,at9578
7,1,G5435,1,1,0,at9578
8,0,G14498,1,1,0,at9762
9,1,G15339,0,0,0,at9762


### Get most dominant GOs

In [13]:
GO_gene=pd.read_csv("output/n5_genes_all/pfam_cat_orf_gene.tsv", sep="\t").set_index("Unnamed: 0")
GO_gene["sum"]=GO_gene.sum(axis=1)
GO_gene_sorted=GO_gene.drop("-").sort_values("sum", ascending=False)
for i, j in zip(GO_gene_sorted.index, GO_gene_sorted["sum"].tolist()):
    print(i+"\t"+str(j)+"\n")

GO:0005524	816.0

GO:0005515	812.0

GO:0016020	518.0

GO:0004672	481.0

GO:0006468	481.0

GO:0003677	242.0

GO:0008270	223.0

GO:0003723	179.0

GO:0006355	162.0

GO:0055085	161.0

GO:0003676	86.0

GO:0005975	45.0

GO:0006508	18.0

GO:0003824	11.0

GO:0016491	11.0

GO:0006886	9.0

GO:0005634	8.0



### Get most dominant Pfams

In [14]:
pfam_gene=pd.read_csv("output/n5_genes_all/pfam_cat_dom_gene.tsv", sep="\t").set_index("Unnamed: 0")
pfam_gene["sum"]=pfam_gene.sum(axis=1)
pfam_gene_sorted=pfam_gene.drop("-").sort_values("sum", ascending=False)
for i, j in zip(pfam_gene_sorted.index, pfam_gene_sorted["sum"].tolist()):
    print(i+"\t"+str(j)+"\n")

Pentatricopeptide repeat	384.0

Protein kinase domain	331.0

Serine-threonine/tyrosine-protein kinase, catalytic domain	219.0

Leucine-rich repeat	171.0

E motif	137.0

Helicase, C-terminal	97.0

RNA recognition motif domain	97.0

Leucine-rich repeat-containing N-terminal, plant-type	67.0

F-box domain	56.0

WD40 repeat	48.0

DYW domain	45.0

SANT/Myb domain	40.0

DEAD/DEAH box helicase domain	37.0

ABC transporter-like, ATP-binding domain	29.0

Zinc finger, RING-type	28.0

C2 domain	14.0

Cytochrome P450	12.0

ATPase, AAA-type, core	11.0

DC1	11.0

Peptidase S8/S53 domain	7.0

Kinesin motor domain	7.0

Peptidase S8 propeptide/proteinase inhibitor I9	6.0

Subtilisin-like protease, fibronectin type-III domain	6.0

Ankyrin repeat	6.0

Proton-dependent oligopeptide transporter family	6.0

Legume lectin domain	5.0

